# Características del Texto (TF-IDF)

Notebook de la etapa **1** del pipeline: análisis exploratorio de características basado en TF-IDF sobre el texto preprocesado de la etapa 00.

> **Nota**: vocabulario referencial, el de deploy lo define la etapa 03.


## Importación de librerías

Se importan las librerías de manipulación de datos (`pandas`, `numpy`) y la de vectorización (`TfidfVectorizer`), que implementa la representación TF-IDF descrita en la siguiente sección.

In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer


## Vectorización TF-IDF

**TF-IDF** (*Term Frequency – Inverse Document Frequency*, frecuencia de término – frecuencia inversa de documento) es una representación numérica del texto: convierte cada documento en un vector donde cada componente es una **frecuencia ponderada** de un término del vocabulario.

### ¿Qué es una frecuencia ponderada?

Una frecuencia simple (como la del BoW) cuenta cuántas veces aparece cada palabra en un texto. El problema: las palabras más comunes del corpus (`school`, `like`, `people`) aparecen en casi todos los documentos y aportan poca información para distinguir ciberacoso de no ciberacoso. La frecuencia ponderada corrige esto penalizando los términos que aparecen en muchos documentos.

El peso TF-IDF de un término $t$ en un documento $d$ es el producto de dos factores:

$$ \mathrm{tfidf}(t,d) = \mathrm{tf}(t,d) \times \mathrm{idf}(t) $$

- **TF** — frecuencia de término: cuántas veces aparece $t$ en $d$ (la frecuencia cruda, que luego se normaliza).
- **IDF** — frecuencia inversa de documento: mide cuán *raro* es $t$ en todo el corpus. Penaliza las palabras que aparecen en casi todos los documentos. Con el suavizado por defecto de scikit-learn:

$$ \mathrm{idf}(t) = \ln\left(\frac{1 + N}{1 + \mathrm{df}(t)}\right) + 1 $$

donde $N$ es el número total de documentos y $\mathrm{df}(t)$ cuántos documentos contienen a $t$. Cuanto más común es $t$, más se acerca $\mathrm{idf}(t)$ a 1; cuanto más raro, más crece.

Resultado: un término recibe peso alto solo si es **frecuente en el documento** y **raro en el corpus** — exactamente lo contrario de las palabras comunes que no discriminan.

En este notebook se ajusta un `TfidfVectorizer` sobre el texto preprocesado con dos parámetros clave:

- `min_df=0.01`: se descartan los términos que aparecen en menos del 1 % de los documentos, eliminando vocabulario extremadamente raro que solo aporta ruido.
- `max_features=1000`: límite superior del tamaño del vocabulario, lo que acota el costo de cómputo y la dimensionalidad de la matriz `X`.

In [2]:

df = pd.read_csv('../data/processed/cyberbullying_preprocessed.csv')

tfidf_vectorizer = TfidfVectorizer(min_df=0.01, max_features=1000)

# Ajustar y transformar los datos
X = tfidf_vectorizer.fit_transform(df['text_preprocessed'])



> **Nota**: el **Bag of Words** es otra técnica de representación, más simple: cuenta la frecuencia de cada palabra sin ponderar. En este proyecto solo se menciona como contraste; la representación usada es TF-IDF.

# Características
X es una matriz dispersa donde cada fila corresponde a un documento y cada columna a una palabra del vocabulario. Los valores son los pesos TF-IDF de cada palabra en cada documento. El X.shape representa la cantidad de documentos y la cantidad de palabras en el vocabulario. Es decir, (n_documentos, n_palabras_vocabulario).


In [4]:
print(X.shape)

(80974, 118)


### Interpretación: reducción drástica del vocabulario

La matriz TF-IDF resultante tiene dimensión **(80 974 × 118)**: de las 38 713 palabras distintas observadas en el corpus, solo **118 términos** sobreviven a los filtros `min_df=0.01` y `max_features=1000`. El filtro de frecuencia mínima descarta todo término que aparece en menos del 1 % de los documentos.


# Calcular la métrica IDF para cada palabra del vocabulario
Este dataset de vocabulario es un diccionario donde la llave es la palabra y el valor es el valor de la métrica IDF. Es decir, (palabra, idf).


In [5]:
features = tfidf_vectorizer.get_feature_names_out()
df_idf = (
    pd.DataFrame({'word': features, 'idf': tfidf_vectorizer.idf_})
    .sort_values('idf', ascending=False)
    .reset_index(drop=True)
)
df_idf


,word,idf
0,dude,5.599935
1,isis,5.597481
2,actually,5.591373
3,head,5.569685
4,boy,5.567304
...,...,...
113,people,3.672088
114,fuck,3.396316
115,like,3.376840
116,school,3.284776


### Interpretación: pesos IDF

El DataFrame muestra los pesos IDF del vocabulario retenido, ordenados de mayor a menor. Un IDF alto significa que el término es **poco frecuente en el corpus** y, por lo tanto, gana más peso relativo cuando aparece en un documento. Un IDF cercano a 1 corresponde a los términos más comunes, que son los menos informativos.

Esto conecta con el hallazgo del EDA: si un término raro pero muy asociado a la clase de bullying aparece en un texto, el modelo le otorga un peso desproporcionado, lo que refuerza el riesgo de falsos positivos. Es un ejemplo concreto de cómo la estadística de frecuencia amplifica sesgos presentes en la etiqueta.

## Resumen del análisis de características

En esta etapa se exploraron las características del texto mediante **TF-IDF**, una **frecuencia ponderada** que combina dos factores:

- **TF** (frecuencia de término): cuántas veces aparece un término en un documento.
- **IDF** (frecuencia inversa de documento): penaliza los términos comunes a todo el corpus.

Con los parámetros:

- `min_df=0.01`: se descartan términos que aparecen en menos del 1 % de los documentos, lo que elimina vocabulario extremadamente raro y reduce ruido.
- `max_features=1000`: límite superior de términos que conforman el vocabulario.

Se inspeccionaron:

- La **matriz TF-IDF** `X` (documentos × términos).
- Los **pesos IDF** de cada palabra del vocabulario.


### Conclusión de la etapa

La etapa de features evidencia que el enfoque **TF-IDF** ofrece una representación útil pero limitada: reduce el texto a frecuencias ponderadas, descarta el orden de las palabras y depende de filtros de frecuencia que pueden dejar fuera justamente el vocabulario discriminante. Estas limitaciones son estructurales del paradigma léxico y se retoman en la discusión de resultados de la etapa 03.
